In [43]:
import numpy as np
import pandas as pd
import joblib
import time
import matplotlib as plt
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score
)

In [25]:
X_train = joblib.load("../data/X_train_zero_day.pkl")
y_train = joblib.load("../data/y_train_zero_day.pkl")

X_train = joblib.load("../data/X_train.pkl")
X_test = joblib.load("../data/X_test.pkl")

y_train = joblib.load("../data/y_train.pkl")
y_test = joblib.load("../data/y_test.pkl")

X_test_seen = joblib.load("../data/X_test_seen.pkl")
y_test_seen = joblib.load("../data/y_test_seen.pkl")

X_test_zero_day = joblib.load("../data/X_test_zero_day.pkl")
y_test_zero_day = joblib.load("../data/y_test_zero_day.pkl")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test_seen:", X_test_seen.shape)
print("y_test_seen:", y_test_seen.shape)

print("X_test_zero_day:", X_test_zero_day.shape)
print("y_test_zero_day:", y_test_zero_day.shape)

X_train: (125973, 122)
y_train: (125973,)
X_test_seen: (18794, 122)
y_test_seen: (18794,)
X_test_zero_day: (3750, 122)
y_test_zero_day: (3750,)


In [3]:
# ============================================
# Keep ONLY normal traffic for unsupervised training
# ============================================

normal_mask = (y_train == 0)

X_train_normal = X_train[normal_mask]

print("Total training samples:", len(X_train))
print("Normal training samples:", len(X_train_normal))
print("Attack training samples:", np.sum(y_train == 1))

Total training samples: 125973
Normal training samples: 67343
Attack training samples: 58630


In [4]:
# ============================================
# Validation split from NORMAL training data
# ============================================

X_normal_train, X_normal_val = train_test_split(
    X_train_normal,
    test_size=0.20,
    random_state=42
)

print("Normal training:", X_normal_train.shape)
print("Normal validation:", X_normal_val.shape)

Normal training: (53874, 122)
Normal validation: (13469, 122)


In [5]:
# ============================================
# Train Isolation Forest
# ============================================

iso_forest = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

iso_forest.fit(X_normal_train)

train_time = time.time() - start_time

print(f"Training completed in {train_time:.2f} seconds")

Training completed in 1.57 seconds


In [6]:
# ============================================
# Calculate anomaly scores
# Higher score = more anomalous
# ============================================

normal_val_scores = -iso_forest.decision_function(X_normal_val)

print("Validation score statistics:")
print("Min :", normal_val_scores.min())
print("Max :", normal_val_scores.max())
print("Mean:", normal_val_scores.mean())
print("Std :", normal_val_scores.std())

Validation score statistics:
Min : -0.1815659435961552
Max : 0.15081871283067994
Mean: -0.1262245895096455
Std : 0.057163863264451865


In [7]:
# ============================================
# Thresholds based on allowed FPR
# ============================================

fpr_targets = [0.01, 0.03, 0.05, 0.10]

thresholds = {}

for fpr in fpr_targets:
    threshold = np.quantile(
        normal_val_scores,
        1 - fpr
    )

    thresholds[fpr] = threshold

    print(
        f"Target FPR <= {fpr*100:.0f}% "
        f"-> Threshold = {threshold:.6f}"
    )

Target FPR <= 1% -> Threshold = 0.032719
Target FPR <= 3% -> Threshold = 0.006295
Target FPR <= 5% -> Threshold = -0.007979
Target FPR <= 10% -> Threshold = -0.033525


In [8]:
# ============================================
# Evaluation function
# ============================================

def evaluate_isolation_forest(
    model,
    X_seen,
    y_seen,
    X_zero_day,
    threshold
):
    
    # Scores
    seen_scores = -model.decision_function(X_seen)
    zero_day_scores = -model.decision_function(X_zero_day)

    # Predictions
    seen_pred = (seen_scores >= threshold).astype(int)
    zero_day_pred = (zero_day_scores >= threshold).astype(int)

    # ----------------------------------------
    # Seen data
    # ----------------------------------------

    cm = confusion_matrix(
        y_seen,
        seen_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) * 100

    seen_f1 = f1_score(
        y_seen,
        seen_pred,
        zero_division=0
    ) * 100

    # ----------------------------------------
    # Zero-Day detection
    # ----------------------------------------

    zero_day_detection = (
        np.sum(zero_day_pred == 1)
        / len(zero_day_pred)
        * 100
    )

    return {
        "Threshold": threshold,
        "Seen F1 (%)": seen_f1,
        "Seen FPR (%)": fpr,
        "Zero-Day Detection (%)": zero_day_detection,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    }

In [9]:
# ============================================
# Evaluate different FPR constraints
# ============================================

results = []

for target_fpr, threshold in thresholds.items():

    result = evaluate_isolation_forest(
        iso_forest,
        X_test_seen,
        y_test_seen,
        X_test_zero_day,
        threshold
    )

    result["Target FPR (%)"] = target_fpr * 100

    results.append(result)


iso_results = pd.DataFrame(results)

iso_results = iso_results[
    [
        "Target FPR (%)",
        "Threshold",
        "Seen F1 (%)",
        "Seen FPR (%)",
        "Zero-Day Detection (%)",
        "TP",
        "FP",
        "TN",
        "FN"
    ]
]

iso_results

,Target FPR (%),Threshold,Seen F1 (%),Seen FPR (%),Zero-Day Detection (%),TP,FP,TN,FN
0,1.0,0.032719,74.736053,1.153331,44.720000,5486,112,9599,3597
1,3.0,0.006295,77.005631,2.059520,57.973333,5812,200,9511,3271
2,5.0,-0.007979,78.223383,2.461127,65.280000,5988,239,9472,3095
3,10.0,-0.033525,79.815951,7.331892,72.400000,6505,712,8999,2578


In [10]:
from sklearn.svm import OneClassSVM

In [11]:
X_ocsvm_train, _ = train_test_split(
    X_normal_train,
    train_size=min(20000, len(X_normal_train)),
    random_state=42
)

print("One-Class SVM training shape:", X_ocsvm_train.shape)

One-Class SVM training shape: (20000, 122)


In [12]:
oc_svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

start_time = time.time()

oc_svm.fit(X_ocsvm_train)

train_time = time.time() - start_time

print(f"One-Class SVM training completed in {train_time:.2f} seconds")

One-Class SVM training completed in 5.50 seconds


In [13]:
ocsvm_val_scores = -oc_svm.decision_function(X_normal_val)

print("One-Class SVM validation score statistics:")
print(f"Min    : {ocsvm_val_scores.min():.6f}")
print(f"Max    : {ocsvm_val_scores.max():.6f}")
print(f"Mean   : {ocsvm_val_scores.mean():.6f}")
print(f"Std    : {ocsvm_val_scores.std():.6f}")

One-Class SVM validation score statistics:
Min    : -5.963852
Max    : 811.601341
Mean   : 5.162509
Std    : 66.082655


In [14]:
fpr_targets = [0.01, 0.03, 0.05, 0.10]

ocsvm_thresholds = {}

for fpr in fpr_targets:
    threshold = np.quantile(ocsvm_val_scores, 1 - fpr)
    ocsvm_thresholds[fpr] = threshold

    print(
        f"Target FPR <= {fpr*100:.0f}% "
        f"-> Threshold = {threshold:.6f}"
    )

Target FPR <= 1% -> Threshold = 70.201144
Target FPR <= 3% -> Threshold = 0.495959
Target FPR <= 5% -> Threshold = -0.000002
Target FPR <= 10% -> Threshold = -0.010442


In [15]:
ocsvm_results = []

for target_fpr, threshold in ocsvm_thresholds.items():

    seen_scores = -oc_svm.decision_function(X_test_seen)
    zero_day_scores = -oc_svm.decision_function(X_test_zero_day)

    seen_pred = (seen_scores >= threshold).astype(int)
    zero_day_pred = (zero_day_scores >= threshold).astype(int)

    cm = confusion_matrix(
        y_test_seen,
        seen_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    seen_fpr = (
        fp / (fp + tn) * 100
        if (fp + tn) > 0 else 0
    )

    seen_f1 = (
        f1_score(
            y_test_seen,
            seen_pred,
            zero_division=0
        ) * 100
    )

    zero_day_detection = (
        np.sum(zero_day_pred == 1)
        / len(zero_day_pred)
        * 100
    )

    ocsvm_results.append({
        "Target FPR (%)": target_fpr * 100,
        "Threshold": threshold,
        "Seen F1 (%)": seen_f1,
        "Seen FPR (%)": seen_fpr,
        "Zero-Day Detection (%)": zero_day_detection,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    })

ocsvm_results = pd.DataFrame(ocsvm_results)

ocsvm_results

,Target FPR (%),Threshold,Seen F1 (%),Seen FPR (%),Zero-Day Detection (%),TP,FP,TN,FN
0,1.0,70.201144,4.850427,0.514880,6.373333,227,50,9661,8856
1,3.0,0.495959,11.903561,2.471424,24.773333,590,240,9471,8493
2,5.0,-0.000002,77.190713,4.242611,43.520000,5968,412,9299,3115
3,10.0,-0.010442,81.316649,5.601895,57.653333,6596,544,9167,2487


In [16]:
from sklearn.neighbors import LocalOutlierFactor

In [17]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
    contamination="auto",
    n_jobs=-1
)

start_time = time.time()

lof.fit(X_ocsvm_train)

train_time = time.time() - start_time

print(f"LOF training completed in {train_time:.2f} seconds")

LOF training completed in 6.18 seconds


In [18]:
lof_val_scores = -lof.decision_function(X_normal_val)

print("LOF validation score statistics:")
print(f"Min    : {lof_val_scores.min():.6f}")
print(f"Max    : {lof_val_scores.max():.6f}")
print(f"Mean   : {lof_val_scores.mean():.6f}")
print(f"Std    : {lof_val_scores.std():.6f}")

LOF validation score statistics:
Min    : -0.588067
Max    : 247353.077434
Mean   : 51.275323
Std    : 3440.628159


In [19]:
lof_thresholds = {}

for fpr in fpr_targets:
    threshold = np.quantile(
        lof_val_scores,
        1 - fpr
    )

    lof_thresholds[fpr] = threshold

    print(
        f"Target FPR <= {fpr*100:.0f}% "
        f"-> Threshold = {threshold:.6f}"
    )

Target FPR <= 1% -> Threshold = 1.423314
Target FPR <= 3% -> Threshold = 0.386676
Target FPR <= 5% -> Threshold = 0.116728
Target FPR <= 10% -> Threshold = -0.163727


In [20]:
def evaluate_lof(
    model,
    X_seen,
    y_seen,
    X_zero_day,
    threshold
):
    seen_scores = -model.decision_function(X_seen)
    zero_day_scores = -model.decision_function(X_zero_day)

    seen_pred = (seen_scores >= threshold).astype(int)
    zero_day_pred = (zero_day_scores >= threshold).astype(int)

    cm = confusion_matrix(
        y_seen,
        seen_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) * 100
    seen_f1 = f1_score(
        y_seen,
        seen_pred,
        zero_division=0
    ) * 100

    zero_day_detection = (
        np.sum(zero_day_pred == 1)
        / len(zero_day_pred)
        * 100
    )

    return {
        "Threshold": threshold,
        "Seen F1 (%)": seen_f1,
        "Seen FPR (%)": fpr,
        "Zero-Day Detection (%)": zero_day_detection,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn
    }


lof_results = []

for target_fpr, threshold in lof_thresholds.items():

    result = evaluate_lof(
        lof,
        X_test_seen,
        y_test_seen,
        X_test_zero_day,
        threshold
    )

    result["Target FPR (%)"] = target_fpr * 100
    lof_results.append(result)


lof_results = pd.DataFrame(lof_results)

lof_results = lof_results[
    [
        "Target FPR (%)",
        "Threshold",
        "Seen F1 (%)",
        "Seen FPR (%)",
        "Zero-Day Detection (%)",
        "TP",
        "FP",
        "TN",
        "FN"
    ]
]

lof_results

,Target FPR (%),Threshold,Seen F1 (%),Seen FPR (%),Zero-Day Detection (%),TP,FP,TN,FN
0,1.0,1.423314,75.943698,6.302132,12.080000,5935,612,9099,3148
1,3.0,0.386676,81.940573,8.958913,16.640000,6908,870,8841,2175
2,5.0,0.116728,84.280173,10.925754,26.693333,7388,1061,8650,1695
3,10.0,-0.163727,85.712738,15.116878,33.173333,7913,1468,8243,1170


In [21]:
import os
import joblib

models_path = "../models"
os.makedirs(models_path, exist_ok=True)

joblib.dump(
    iso_forest,
    f"{models_path}/isolation_forest.pkl"
)

joblib.dump(
    oc_svm,
    f"{models_path}/one_class_svm.pkl"
)

joblib.dump(
    lof,
    f"{models_path}/local_outlier_factor.pkl"
)

unsupervised_thresholds = {
    "isolation_forest": thresholds[0.05],
    "one_class_svm": ocsvm_thresholds[0.05],
    "local_outlier_factor": lof_thresholds[0.05]
}

joblib.dump(
    unsupervised_thresholds,
    f"{models_path}/unsupervised_thresholds.pkl"
)

print("Unsupervised models saved successfully.")

Unsupervised models saved successfully.


In [32]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [69]:
def evaluate_anomaly_model(model, X_train, X_test, y_test):

    start_train = time.time()

    model.fit(X_train)

    train_time = time.time() - start_train

    start_predict = time.time()
    y_pred_raw = model.predict(X_test)

    predict_time = time.time() - start_predict

    # -1 = Attack
    #  1 = Normal

    y_pred = np.where(y_pred_raw == -1, 1, 0)

    accuracy = accuracy_score(y_test, y_pred)

    report = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    precision = report["1"]["precision"]
    recall = report["1"]["recall"]
    f1 = report["1"]["f1-score"]

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    print("=" * 50)
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"FPR      : {fpr:.4f}")

    print(f"Training Time   : {train_time:.4f} seconds")
    print(f"Prediction Time : {predict_time:.4f} seconds")

    print("\nClassification Report:")

    print(
        classification_report(
            y_test,
            y_pred,
            target_names=["Normal", "Attack"],
            zero_division=0
        )
    )


    return {
        "model": model.__class__.__name__,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "fpr": fpr,
        "train_time": train_time,
        "predict_time": predict_time
    }

In [74]:
isolation_forest = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42,
    n_jobs=-1
)

if_result = evaluate_anomaly_model(
    isolation_forest,
    X_train_scaled,
    X_test_scaled,
    y_test
)

Model: IsolationForest
Accuracy : 0.5530
Precision: 0.8722
Recall   : 0.2516
F1-Score : 0.3906
FPR      : 0.0487
Training Time   : 1.8661 seconds
Prediction Time : 0.2324 seconds

Classification Report:
              precision    recall  f1-score   support

      Normal       0.49      0.95      0.65      9711
      Attack       0.87      0.25      0.39     12833

    accuracy                           0.55     22544
   macro avg       0.68      0.60      0.52     22544
weighted avg       0.71      0.55      0.50     22544



In [62]:
one_class_svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

ocsvm_result = evaluate_anomaly_model(
    one_class_svm,
    X_train_scaled,
    X_test_scaled,
    y_test
)

Model: OneClassSVM
Accuracy : 0.5577
Precision: 0.9667
Recall   : 0.2309
F1-Score : 0.3728
FPR      : 0.0105
Training Time   : 1428.7889 seconds
Prediction Time : 29.2926 seconds

Classification Report:
              precision    recall  f1-score   support

      Normal       0.49      0.99      0.66      9711
      Attack       0.97      0.23      0.37     12833

    accuracy                           0.56     22544
   macro avg       0.73      0.61      0.52     22544
weighted avg       0.76      0.56      0.50     22544



In [63]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.1,
    novelty=True,
    n_jobs=-1
)

lof_result = evaluate_anomaly_model(
    lof,
    X_train_scaled,
    X_test_scaled,
    y_test
)

Model: LocalOutlierFactor
Accuracy : 0.6245
Precision: 0.7406
Recall   : 0.5238
F1-Score : 0.6136
FPR      : 0.2424
Training Time   : 117.8288 seconds
Prediction Time : 19.4566 seconds

Classification Report:
              precision    recall  f1-score   support

      Normal       0.55      0.76      0.63      9711
      Attack       0.74      0.52      0.61     12833

    accuracy                           0.62     22544
   macro avg       0.64      0.64      0.62     22544
weighted avg       0.66      0.62      0.62     22544



In [75]:
results = [
    if_result,
    ocsvm_result,
    lof_result
]

import pandas as pd

results_df = pd.DataFrame(results)

results_df

,model,accuracy,precision,recall,f1_score,fpr,train_time,predict_time
0,IsolationForest,0.553007,0.872231,0.251617,0.390565,0.048708,1.866066,0.232412
1,OneClassSVM,0.557665,0.966721,0.230889,0.372751,0.010504,1428.788939,29.292622
2,LocalOutlierFactor,0.624512,0.740635,0.523806,0.613629,0.242406,117.828761,19.456550
